In [1]:
!python -m traffic_classifier samples/web.pcapng --csv output/web_features.csv

capture: samples/web.pcapng
parsed_packets: 42749
flows: 96
csv: output/web_features.csv

  1. Web        confidence=0.88 packets=15   bytes=2115   121.36.38.147:443 <-> 172.27.152.109:50926 TCP (HTTP/HTTPS port)
  2. Web        confidence=0.88 packets=5    bytes=260    172.27.152.109:55124 <-> 216.239.36.223:443 TCP (HTTP/HTTPS port)
  3. Web        confidence=0.88 packets=5    bytes=260    172.27.152.109:55125 <-> 216.239.36.223:443 TCP (HTTP/HTTPS port)
  4. Web        confidence=0.88 packets=19   bytes=7186   157.148.41.176:443 <-> 172.27.152.109:55586 TCP (HTTP/HTTPS port)
  5. Web        confidence=0.88 packets=140  bytes=64525  116.181.3.98:443 <-> 172.27.152.109:54789 TCP (HTTP/HTTPS port)
  6. Web        confidence=0.88 packets=3    bytes=337    172.27.152.109:57516 <-> 4.145.79.81:443 TCP (HTTP/HTTPS port)
  7. Web        confidence=0.88 packets=65   bytes=36988  157.148.41.176:443 <-> 172.27.152.109:55587 TCP (HTTP/HTTPS port)
  8. Web        confidence=0.88 packets=29   byt

In [2]:
!ls -lh traffic_classifier

total 64K
-rw-r--r-- 1 nixos users  964 May 26 12:07 analyzer.py
-rw-r--r-- 1 nixos users 3.2K May 26 11:41 classifier.py
-rw-r--r-- 1 nixos users  561 May 20 12:27 csv_io.py
-rw-r--r-- 1 nixos users 2.3K May 20 12:27 features.py
-rw-r--r-- 1 nixos users  555 May 20 12:27 flow.py
-rw-r--r-- 1 nixos users   73 May 20 12:27 __init__.py
-rw-r--r-- 1 nixos users 3.8K May 26 12:13 live_capture.py
-rw-r--r-- 1 nixos users 1.7K May 26 11:42 __main__.py
-rw-r--r-- 1 nixos users 5.3K May 26 12:43 ml.py
-rw-r--r-- 1 nixos users 1.7K May 20 12:27 models.py
-rw-r--r-- 1 nixos users 3.7K May 20 12:27 parser.py
-rw-r--r-- 1 nixos users 4.0K May 20 12:27 pcap_reader.py
-rw-r--r-- 1 nixos users 1.7K May 26 12:08 predict.py
drwxr-xr-x 2 nixos users 4.0K May 26 12:44 __pycache__
-rw-r--r-- 1 nixos users 1.6K May 26 12:43 train.py


In [3]:
!python -m traffic_classifier.train output/dns_features.csv output/icmp_features.csv output/web_features.csv --model models/knn_model.json

model: models/knn_model.json
samples: 342
labels: DNS, ICMP, Web
k: 3


In [4]:
import csv

  cols = [
      "flow_id",
      "protocol",
      "packet_count",
      "avg_packet_size",
      "interval_variance",
      "label",
      "confidence",
  ]

  with open("output/web_features.csv", encoding="utf-8") as f:
      rows = list(csv.DictReader(f))[:5]

  print("\t".join(cols))
  for row in rows:
      print("\t".join(str(row.get(col, "")) for col in cols))

IndentationError: unexpected indent (1827046620.py, line 3)

In [5]:
import csv

cols = [
    "flow_id",
    "protocol",
    "packet_count",
    "avg_packet_size",
    "interval_variance",
    "label",
    "confidence",
]

with open("output/web_features.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))[:5]

print("\t".join(cols))
for row in rows:
    print("\t".join(str(row.get(col, "")) for col in cols))

flow_id	protocol	packet_count	avg_packet_size	interval_variance	label	confidence
121.36.38.147:443 <-> 172.27.152.109:50926 TCP	TCP	15	141.0	84.676853	Web	0.88
172.27.152.109:55124 <-> 216.239.36.223:443 TCP	TCP	5	52.0	7.218891	Web	0.88
172.27.152.109:55125 <-> 216.239.36.223:443 TCP	TCP	5	52.0	7.179729	Web	0.88
157.148.41.176:443 <-> 172.27.152.109:55586 TCP	TCP	19	378.210526	120.230934	Web	0.88
116.181.3.98:443 <-> 172.27.152.109:54789 TCP	TCP	140	460.892857	1.887795	Web	0.88


In [6]:
!python -m traffic_classifier.predict samples/live_100.pcapng --model models/knn_model.json --limit 10

capture: samples/live_100.pcapng
model: models/knn_model.json
parsed_packets: 105
flows: 19

  1. ML=Web        confidence=1.00 rule=Web        packets=2    121.36.38.147:443 <-> 172.27.152.109:50926 TCP
  2. ML=Web        confidence=1.00 rule=Web        packets=18   157.148.41.176:443 <-> 172.27.152.109:55587 TCP
  3. ML=Web        confidence=1.00 rule=Web        packets=2    172.27.152.109:57338 <-> 216.239.34.223:443 TCP
  4. ML=Web        confidence=1.00 rule=Web        packets=5    116.162.46.244:443 <-> 172.27.152.109:53649 TCP
  5. ML=Web        confidence=1.00 rule=Unknown    packets=6    112.86.230.139:14000 <-> 172.27.152.109:63054 TCP
  6. ML=Web        confidence=1.00 rule=Small_UDP  packets=3    172.27.200.37:5353 <-> 224.0.0.251:5353 UDP
  7. ML=Web        confidence=1.00 rule=Small_UDP  packets=2    172.27.200.37:59953 <-> 224.0.0.252:5355 UDP
  8. ML=Web        confidence=1.00 rule=Web        packets=4    157.148.41.176:443 <-> 172.27.152.109:55586 TCP
  9. ML=DNS      

In [7]:
!python -m traffic_classifier samples/false_positive_port_scan.pcap

capture: samples/false_positive_port_scan.pcap
parsed_packets: 5
flows: 5

  1. ALERT Port_Scan  confidence=0.80 packets=1    bytes=40     10.0.0.2:50000 <-> 10.0.0.3:8001 TCP (TCP probes to 5 destination ports)
  2. ALERT Port_Scan  confidence=0.80 packets=1    bytes=40     10.0.0.2:50001 <-> 10.0.0.3:8002 TCP (TCP probes to 5 destination ports)
  3. ALERT Port_Scan  confidence=0.80 packets=1    bytes=40     10.0.0.2:50002 <-> 10.0.0.3:8003 TCP (TCP probes to 5 destination ports)
  4. ALERT Port_Scan  confidence=0.80 packets=1    bytes=40     10.0.0.2:50003 <-> 10.0.0.3:8004 TCP (TCP probes to 5 destination ports)
  5. ALERT Port_Scan  confidence=0.80 packets=1    bytes=40     10.0.0.2:50004 <-> 10.0.0.3:8005 TCP (TCP probes to 5 destination ports)


In [8]:
!python -m pytest tests

/nix/store/sd81bvmch7njdpwx3lkjslixcbj5mivz-python3-3.13.4/bin/python: No module named pytest


In [9]:
!python -m pytest tests

/nix/store/sd81bvmch7njdpwx3lkjslixcbj5mivz-python3-3.13.4/bin/python: No module named pytest


In [10]:
  %cd /home/nixos/workspace/C-Demo/CPE/traffic_classifier

  from pathlib import Path
  import tempfile
  import importlib.util
  import traceback

  test_file = Path("tests/test_pipeline.py")
  spec = importlib.util.spec_from_file_location("test_pipeline", test_file)
  mod = importlib.util.module_from_spec(spec)
  spec.loader.exec_module(mod)

  tests = [
      mod.test_web_flow_rule_classification,
      mod.test_dns_flow_rule_classification,
      mod.test_syn_flood_classification_raises_alert_label,
      mod.test_short_tcp_probe_raises_alert_label,
      mod.test_repeated_tcp_probes_to_many_ports_mark_port_scan,
  ]

  passed = 0

  for test in tests:
      test()
      print(f"PASS {test.__name__}")
      passed += 1

  with tempfile.TemporaryDirectory() as tmp:
      mod.test_knn_model_predicts_labeled_rows(Path(tmp))
      print("PASS test_knn_model_predicts_labeled_rows")
      passed += 1

  print()
  print(f"pipeline tests passed: {passed}/6")

/home/nixos/workspace/C-Demo/CPE/traffic_classifier
PASS test_web_flow_rule_classification
PASS test_dns_flow_rule_classification
PASS test_syn_flood_classification_raises_alert_label
PASS test_short_tcp_probe_raises_alert_label
PASS test_repeated_tcp_probes_to_many_ports_mark_port_scan
PASS test_knn_model_predicts_labeled_rows

pipeline tests passed: 6/6


In [11]:
!python -m pytest tests

/nix/store/sd81bvmch7njdpwx3lkjslixcbj5mivz-python3-3.13.4/bin/python: No module named pytest


In [12]:
 !python -m pytest tests

  print("""1. python -m traffic_classifier samples/dns.pcapng --csv output/dns_features.csv
  2. python -m traffic_classifier.train output/dns_features.csv output/icmp_features.csv output/web_features.csv --model
  models/knn_model.json
  3. python -m traffic_classifier.predict samples/web.pcapng --model models/knn_model.json --limit 5
  4. python -m traffic_classifier.predict samples/live_100.pcapng --model models/knn_model.json --limit 10
  5. python -m traffic_classifier samples/false_positive_port_scan.pcap""")

IndentationError: unexpected indent (3144350086.py, line 3)

In [13]:
!python -m pytest tests

print("""1. python -m traffic_classifier samples/dns.pcapng --csv output/dns_features.csv
2. python -m traffic_classifier.train output/dns_features.csv output/icmp_features.csv output/web_features.csv --model
models/knn_model.json
3. python -m traffic_classifier.predict samples/web.pcapng --model models/knn_model.json --limit 5
4. python -m traffic_classifier.predict samples/live_100.pcapng --model models/knn_model.json --limit 10
5. python -m traffic_classifier samples/false_positive_port_scan.pcap""")

/nix/store/sd81bvmch7njdpwx3lkjslixcbj5mivz-python3-3.13.4/bin/python: No module named pytest
1. python -m traffic_classifier samples/dns.pcapng --csv output/dns_features.csv
2. python -m traffic_classifier.train output/dns_features.csv output/icmp_features.csv output/web_features.csv --model
models/knn_model.json
3. python -m traffic_classifier.predict samples/web.pcapng --model models/knn_model.json --limit 5
4. python -m traffic_classifier.predict samples/live_100.pcapng --model models/knn_model.json --limit 10
5. python -m traffic_classifier samples/false_positive_port_scan.pcap


In [14]:
!python -m traffic_classifier.live_capture -i eth0 --count 100 --model models/knn_model.json

capturing 100 packets on eth0 ...
error: [Errno 1] Operation not permitted
hint: live capture usually needs root or CAP_NET_RAW permission


In [15]:
!python -m traffic_classifier.predict samples/web.pcapng --model models/knn_model.json --limit 8

capture: samples/web.pcapng
model: models/knn_model.json
parsed_packets: 42749
flows: 96

  1. ML=Web        confidence=1.00 rule=Web        packets=15   121.36.38.147:443 <-> 172.27.152.109:50926 TCP
  2. ML=Web        confidence=1.00 rule=Web        packets=5    172.27.152.109:55124 <-> 216.239.36.223:443 TCP
  3. ML=Web        confidence=1.00 rule=Web        packets=5    172.27.152.109:55125 <-> 216.239.36.223:443 TCP
  4. ML=Web        confidence=1.00 rule=Web        packets=19   157.148.41.176:443 <-> 172.27.152.109:55586 TCP
  5. ML=Web        confidence=1.00 rule=Web        packets=140  116.181.3.98:443 <-> 172.27.152.109:54789 TCP
  6. ML=Web        confidence=1.00 rule=Web        packets=3    172.27.152.109:57516 <-> 4.145.79.81:443 TCP
  7. ML=Web        confidence=1.00 rule=Web        packets=65   157.148.41.176:443 <-> 172.27.152.109:55587 TCP
  8. ML=Web        confidence=1.00 rule=Web        packets=29   172.27.152.109:53743 <-> 20.205.243.168:443 TCP
... 88 more flows om

In [16]:
print("""现场演示命令顺序

1. 离线分析 pcap，导出 CSV
python -m traffic_classifier samples/dns.pcapng --csv output/dns_features.csv

2. 训练 KNN 模型
python -m traffic_classifier.train output/dns_features.csv output/icmp_features.csv output/web_features.csv --model models/
knn_model.json

3. 加载模型预测 Web 样本
python -m traffic_classifier.predict samples/web.pcapng --model models/knn_model.json --limit 5

4. 读取 100 包样本并预测
python -m traffic_classifier.predict samples/live_100.pcapng --model models/knn_model.json --limit 10

5. 展示攻击报警误报样本
python -m traffic_classifier samples/false_positive_port_scan.pcap""")

现场演示命令顺序

1. 离线分析 pcap，导出 CSV
python -m traffic_classifier samples/dns.pcapng --csv output/dns_features.csv

2. 训练 KNN 模型
python -m traffic_classifier.train output/dns_features.csv output/icmp_features.csv output/web_features.csv --model models/
knn_model.json

3. 加载模型预测 Web 样本
python -m traffic_classifier.predict samples/web.pcapng --model models/knn_model.json --limit 5

4. 读取 100 包样本并预测
python -m traffic_classifier.predict samples/live_100.pcapng --model models/knn_model.json --limit 10

5. 展示攻击报警误报样本
python -m traffic_classifier samples/false_positive_port_scan.pcap
